# Model Ablation Comparison (Clean Rewrite)

This notebook evaluates selected Uniform/Zonal ablations with consistent model filtering (excluding `Full`) and emphasizes raw-life error metrics for `LogLife`.


In [ ]:
from __future__ import annotations

import ast
import contextlib
import importlib.util
import os
import sys
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import Markdown, display

try:
    import torch
except Exception:
    torch = None

try:
    import h5py
except Exception:
    h5py = None

try:
    from sklearn.model_selection import train_test_split
except Exception:
    train_test_split = None

pd.set_option('display.max_colwidth', 160)


In [ ]:
# Global config (defined once)
CURRENT_DIR = Path(os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd())
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / 'Uniform').exists() else CURRENT_DIR.parent
CHECKPOINT_ROOT = REPO_ROOT
H5_DATASET_ROOT = REPO_ROOT / 'Data_gen' / 'output'
SPLIT_SEED = 42
SPLIT_SIZE = 0.2

EXAMPLE_DIR_CANDIDATES = [
    Path('/home/runner/Desktop/Disc_lifing_paper/Comparison/Examples'),
    Path('/Desktop/Disc_lifing_paper/Comparison/Examples'),
    REPO_ROOT / 'Comparison' / 'Examples',
]

TARGET_FAMILIES = ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint', 'PointNetMLPJoint_headfeat']
MAIN_FAMILIES = ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint']
MODEL_COLORS = {
    'ArGEnT_self_att_noSDF': '#1f77b4',
    'PointNetMLPJoint': '#ff7f0e',
    'PointNetMLPJoint_headfeat': '#2ca02c',
}

if not (REPO_ROOT / 'Uniform').exists() or not (REPO_ROOT / 'Zonal').exists():
    raise RuntimeError(f'Could not locate repository root from {CURRENT_DIR}')


In [ ]:
def map_target_name(name: str) -> str:
    s = str(name).strip().lower()
    if 'stress' in s:
        return 'Stress'
    if 'life' in s:
        return 'LogLife'
    return str(name)


def parse_training_constants(training_script: Path) -> Dict[str, Any]:
    if not training_script.exists():
        return {}
    tree = ast.parse(training_script.read_text(encoding='utf-8'))
    wanted = {'TARGET_NAMES', 'INPUT_COLS', 'QUERY_COLS', 'H5_FILENAME'}
    out: Dict[str, Any] = {}
    for node in tree.body:
        if isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id in wanted and node.value is not None:
            try:
                out[node.target.id] = ast.literal_eval(node.value)
            except Exception:
                pass
        elif isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in wanted:
                    try:
                        out[target.id] = ast.literal_eval(node.value)
                    except Exception:
                        pass
    return out


def discover_model_rows() -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for regime in ['Uniform', 'Zonal']:
        regime_dir = CHECKPOINT_ROOT / regime
        if not regime_dir.exists():
            continue
        for ablation_dir in sorted([p for p in regime_dir.iterdir() if p.is_dir()]):
            ablation = ablation_dir.name
            if ablation.lower() == 'full':
                continue
            for model_dir in sorted([p for p in ablation_dir.iterdir() if p.is_dir()]):
                model_family = model_dir.name
                if model_family not in TARGET_FAMILIES:
                    continue
                ts_path = model_dir / 'Training_script.py'
                cfg = parse_training_constants(ts_path)
                ckpts = sorted((model_dir / 'Trained_models').glob('*.pt')) if (model_dir / 'Trained_models').exists() else []
                rows.append({
                    'regime': regime,
                    'ablation': ablation,
                    'model_family': model_family,
                    'training_dir': model_dir,
                    'training_script': ts_path,
                    'checkpoint_path': ckpts[0] if ckpts else None,
                    'target_names': [map_target_name(t) for t in cfg.get('TARGET_NAMES', [])],
                    'input_cols': cfg.get('INPUT_COLS', [0, 1]),
                    'query_cols': cfg.get('QUERY_COLS', [0, 1]),
                    'h5_filename': cfg.get('H5_FILENAME'),
                })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(['regime', 'ablation', 'model_family']).reset_index(drop=True)


def load_h5_pointsets(path: Path) -> List[np.ndarray]:
    sets: List[np.ndarray] = []
    with h5py.File(path, 'r') as f:
        if 'samples' not in f:
            raise RuntimeError(f"'samples' group missing in {path}")
        for key in sorted(f['samples'].keys()):
            grp = f['samples'][key]
            coords = grp['node_coords_mm'][:].astype(np.float32)
            zone_id = grp['zone_id'][:].astype(np.float32)
            stress = grp['stress_max_vm'][:].astype(np.float32)
            life_raw = grp['life_raw'][:].astype(np.float32)
            cols: List[np.ndarray] = [coords[:, 0], coords[:, 1], zone_id]
            if 'arc_length_mm' in grp:
                arc = grp['arc_length_mm'][:].astype(np.float32)
                if not np.any(np.isnan(arc)):
                    cols.append(arc)
            if 'node_features' in grp:
                feats = grp['node_features'][:]
                if feats.ndim == 2 and feats.shape[1] > 0:
                    for i in range(feats.shape[1]):
                        cols.append(feats[:, i].astype(np.float32))
            cols.append(stress)
            cols.append(np.log10(np.clip(life_raw, 1e-12, None)).astype(np.float32))
            sets.append(np.stack(cols, axis=-1).astype(np.float32))
    return sets


def split_indices(n: int, test_size: float = SPLIT_SIZE, random_state: int = SPLIT_SEED) -> Tuple[np.ndarray, np.ndarray]:
    idx = np.arange(n)
    if train_test_split is not None:
        tr, va = train_test_split(idx, test_size=test_size, random_state=random_state)
        return np.asarray(tr, dtype=int), np.asarray(va, dtype=int)
    rng = np.random.default_rng(random_state)
    perm = rng.permutation(idx)
    n_val = max(1, int(round(n * test_size)))
    return perm[n_val:], perm[:n_val]


def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean((y_true - y_pred) ** 2))


def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))


def r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    if ss_tot <= 1e-12:
        return np.nan
    return float(1.0 - ss_res / ss_tot)


def loglife_to_raw_life(loglife: np.ndarray) -> np.ndarray:
    x = np.asarray(loglife, dtype=np.float64)
    return np.power(10.0, np.clip(x, -30.0, 30.0))


def percentage_metrics(y_true_raw: np.ndarray, y_pred_raw: np.ndarray) -> Dict[str, float]:
    y_true_raw = np.asarray(y_true_raw, dtype=np.float64)
    y_pred_raw = np.asarray(y_pred_raw, dtype=np.float64)
    denom = np.clip(y_true_raw, 1e-12, None)
    pe = (y_pred_raw - y_true_raw) / denom * 100.0
    ape = np.abs(pe)
    return {
        'MAPE (%)': float(np.mean(ape)),
        'MPE (%)': float(np.mean(pe)),
        'Max_PE (%)': float(np.max(ape)),
    }


In [ ]:
@contextlib.contextmanager
def prepend_sys_path(path: Path):
    p = str(path)
    sys.path.insert(0, p)
    try:
        yield
    finally:
        if p in sys.path:
            sys.path.remove(p)


def import_module_from_file(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, str(file_path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot import {file_path}')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def to_numpy(x):
    if torch is not None and isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def denorm_targets(pred_norm: np.ndarray, ckpt: Dict[str, Any], n_targets: int) -> np.ndarray:
    target_mean = to_numpy(ckpt['target_mean']).reshape(1, -1)
    target_std = to_numpy(ckpt['target_std']).reshape(1, -1)
    if target_mean.shape[1] < n_targets:
        raise RuntimeError('target_mean has fewer entries than target count')
    return pred_norm[:, :n_targets] * target_std[:, :n_targets] + target_mean[:, :n_targets]


def normalize_cols(arr: np.ndarray, cols: List[int], ckpt: Dict[str, Any]) -> np.ndarray:
    coord_center = to_numpy(ckpt['coord_center']).reshape(-1)
    coord_half_range = np.clip(to_numpy(ckpt['coord_half_range']).reshape(-1), 1e-12, None)
    extra_stats = ckpt.get('extra_feat_stats') or {}

    out = np.zeros((arr.shape[0], len(cols)), dtype=np.float32)
    for i, c in enumerate(cols):
        col = arr[:, c].astype(np.float32)
        if c == 0:
            out[:, i] = (col - float(coord_center[0])) / float(coord_half_range[0])
        elif c == 1:
            out[:, i] = (col - float(coord_center[1])) / float(coord_half_range[1])
        elif c in extra_stats:
            mu = float(extra_stats[c].get('mean', 0.0))
            sd = max(float(extra_stats[c].get('std', 1.0)), 1e-6)
            out[:, i] = (col - mu) / sd
        else:
            out[:, i] = col
    return out


def reconstruct_model_from_checkpoint(training_dir: Path, model_family: str, ckpt: Dict[str, Any]):
    arch = ckpt.get('arch')
    if not isinstance(arch, dict):
        raise RuntimeError('checkpoint arch missing or invalid')

    with prepend_sys_path(training_dir):
        if model_family.startswith('PointNetMLPJoint'):
            pn = import_module_from_file(f'pn_models_{training_dir.name}', training_dir / 'pn_models.py')
            if hasattr(pn, 'build_model_from_arch'):
                model = pn.build_model_from_arch(arch)
            else:
                model = pn.PointNetMLPJoint(
                    latent_dim=int(arch.get('encoder_cfg', {}).get('latent_dim', 128)),
                    mlp_hidden=arch.get('head_hidden', [256, 256, 128]),
                    out_dim=int(arch.get('out_dim', 1)),
                    encoder_cfg=arch.get('encoder_cfg', None),
                    in_channels=int(arch.get('encoder_cfg', {}).get('in_channels', 0)),
                )
        elif model_family == 'ArGEnT_self_att_noSDF':
            bench = import_module_from_file(f'benchmarks_{training_dir.name}', training_dir / 'benchmarks.py')
            model = bench.ArGEnTDeepONet(**{k: v for k, v in arch.items() if k in bench.ArGEnTDeepONet.__init__.__code__.co_varnames})
        else:
            raise RuntimeError(f'Unsupported model family: {model_family}')

    model.load_state_dict(ckpt['model_state'], strict=True)
    model.eval()
    return model


MODEL_CACHE: Dict[str, Any] = {}
CKPT_CACHE: Dict[str, Dict[str, Any]] = {}


def get_model_and_ckpt(row: pd.Series):
    ckpt_path = row['checkpoint_path']
    if ckpt_path is None or not Path(ckpt_path).exists():
        raise RuntimeError('checkpoint file missing')

    key = str(ckpt_path)
    if key not in CKPT_CACHE:
        CKPT_CACHE[key] = torch.load(ckpt_path, map_location='cpu')
    ckpt = CKPT_CACHE[key]

    if key not in MODEL_CACHE:
        MODEL_CACHE[key] = reconstruct_model_from_checkpoint(Path(row['training_dir']), row['model_family'], ckpt)
    model = MODEL_CACHE[key]
    return model, ckpt


def run_model_on_array(arr: np.ndarray, row: pd.Series) -> Tuple[np.ndarray, np.ndarray, List[str], Dict[str, Any]]:
    model, ckpt = get_model_and_ckpt(row)

    target_names = row.get('target_names') or []
    if not target_names:
        n_targets = int(to_numpy(ckpt['target_mean']).reshape(-1).shape[0])
        target_names = [f'Target_{i}' for i in range(n_targets)]
    n_targets = len(target_names)

    true_targets = arr[:, -n_targets:]
    input_cols = list(row.get('input_cols', [0, 1]))
    query_cols = list(row.get('query_cols', [0, 1]))
    extra_feat_cols = ckpt.get('extra_feat_cols')
    head_feat_cols = ckpt.get('head_feat_cols')

    if row['model_family'] == 'ArGEnT_self_att_noSDF':
        enc = normalize_cols(arr, input_cols, ckpt)
        qry = normalize_cols(arr, query_cols, ckpt)
        with torch.no_grad():
            pred_norm = model(
                torch.tensor(enc, dtype=torch.float32).unsqueeze(0),
                torch.tensor(qry, dtype=torch.float32).unsqueeze(0),
            ).squeeze(0).cpu().numpy()
    elif row['model_family'] == 'PointNetMLPJoint_headfeat':
        feature_cols = list(extra_feat_cols) if isinstance(extra_feat_cols, list) else list(range(2, arr.shape[1] - n_targets))
        hf_cols = list(head_feat_cols) if isinstance(head_feat_cols, list) else feature_cols
        xy = normalize_cols(arr, [0, 1], ckpt)
        geom = normalize_cols(arr, feature_cols, ckpt)
        head = normalize_cols(arr, hf_cols, ckpt)
        with torch.no_grad():
            pred_norm = model(
                torch.tensor(xy, dtype=torch.float32).unsqueeze(0),
                torch.tensor(xy, dtype=torch.float32).unsqueeze(0),
                torch.tensor(geom, dtype=torch.float32).unsqueeze(0),
                torch.tensor(head, dtype=torch.float32).unsqueeze(0),
            ).squeeze(0).cpu().numpy()
    else:
        feature_cols = list(extra_feat_cols) if isinstance(extra_feat_cols, list) else list(range(2, arr.shape[1] - n_targets))
        xy = normalize_cols(arr, [0, 1], ckpt)
        geom = normalize_cols(arr, feature_cols, ckpt)
        with torch.no_grad():
            pred_norm = model(
                torch.tensor(xy, dtype=torch.float32).unsqueeze(0),
                torch.tensor(xy, dtype=torch.float32).unsqueeze(0),
                torch.tensor(geom, dtype=torch.float32).unsqueeze(0),
            ).squeeze(0).cpu().numpy()

    pred_targets = denorm_targets(pred_norm, ckpt, n_targets)
    return pred_targets, true_targets, target_names, ckpt


def evaluate_all_models() -> pd.DataFrame:
    rows = discover_model_rows()
    if rows.empty:
        return pd.DataFrame()
    if torch is None or h5py is None:
        warnings.warn('Skipping evaluation because torch and/or h5py are unavailable in this runtime.')
        return pd.DataFrame()

    records: List[Dict[str, Any]] = []
    split_cache: Dict[Tuple[str, int], np.ndarray] = {}

    for _, row in rows.iterrows():
        if not row['h5_filename']:
            continue
        h5_path = H5_DATASET_ROOT / row['h5_filename']
        if not h5_path.exists():
            continue

        try:
            pointsets = load_h5_pointsets(h5_path)
            key = (str(h5_path), len(pointsets))
            if key not in split_cache:
                _, val_idx = split_indices(len(pointsets), test_size=SPLIT_SIZE, random_state=SPLIT_SEED)
                split_cache[key] = np.asarray(sorted(val_idx.tolist()), dtype=int)
            val_sets = [pointsets[i] for i in split_cache[key]]

            preds, trues = [], []
            target_names: List[str] = []
            for arr in val_sets:
                pred, true, target_names, _ = run_model_on_array(arr, row)
                preds.append(pred)
                trues.append(true)

            y_pred = np.concatenate(preds, axis=0)
            y_true = np.concatenate(trues, axis=0)

            for j, target in enumerate(target_names):
                target_std = map_target_name(target)
                y_t = y_true[:, j]
                y_p = y_pred[:, j]
                rec: Dict[str, Any] = {
                    'regime': row['regime'],
                    'ablation': row['ablation'],
                    'model_family': row['model_family'],
                    'target': target_std,
                    'MSE': mse(y_t, y_p),
                    'RMSE': rmse(y_t, y_p),
                    'MAE': mae(y_t, y_p),
                    'R2 (log)': r2(y_t, y_p),
                    'MAPE (%)': np.nan,
                    'MPE (%)': np.nan,
                    'Max_PE (%)': np.nan,
                    'RMSE_raw': np.nan,
                    'MSE_raw_life': np.nan,
                    'R2_raw_life': np.nan,
                }

                if target_std == 'LogLife':
                    y_t_raw = loglife_to_raw_life(y_t)
                    y_p_raw = loglife_to_raw_life(y_p)
                    raw_mse = mse(y_t_raw, y_p_raw)
                    rec.update(percentage_metrics(y_t_raw, y_p_raw))
                    rec['MSE_raw_life'] = raw_mse
                    rec['RMSE_raw'] = float(np.sqrt(raw_mse))
                    rec['R2_raw_life'] = r2(y_t_raw, y_p_raw)

                records.append(rec)
        except Exception as exc:
            warnings.warn(f"Skipping {row['regime']}/{row['ablation']}/{row['model_family']}: {type(exc).__name__}: {exc}")

    out = pd.DataFrame(records)
    if out.empty:
        return out
    return out.sort_values(['regime', 'ablation', 'model_family', 'target']).reset_index(drop=True)


selected_models_df = discover_model_rows()
metrics_long = evaluate_all_models()

display(selected_models_df[['regime', 'ablation', 'model_family', 'checkpoint_path', 'h5_filename']] if not selected_models_df.empty else pd.DataFrame())


In [ ]:
def get_study_results(
    metrics_df: pd.DataFrame,
    regimes: List[str],
    ablations: List[str],
    models: List[str],
) -> pd.DataFrame:
    if metrics_df.empty:
        return pd.DataFrame()
    out = metrics_df[
        metrics_df['regime'].isin(regimes)
        & metrics_df['ablation'].isin(ablations)
        & metrics_df['model_family'].isin(models)
    ].copy()
    if out.empty:
        return out
    out['ablation'] = pd.Categorical(out['ablation'], categories=ablations, ordered=True)
    out['model_family'] = pd.Categorical(out['model_family'], categories=models, ordered=True)
    out['regime'] = pd.Categorical(out['regime'], categories=regimes, ordered=True)
    return out.sort_values(['ablation', 'model_family', 'regime', 'target']).reset_index(drop=True)


def render_study_table(study_df: pd.DataFrame, title: str):
    if study_df.empty:
        display(Markdown(f"**{title}: no evaluated rows available in this runtime.**"))
        return

    cols = [
        'regime', 'ablation', 'model_family', 'target',
        'MAPE (%)', 'MPE (%)', 'Max_PE (%)', 'RMSE_raw', 'MSE_raw_life', 'R2_raw_life',
        'MSE', 'RMSE', 'MAE', 'R2 (log)'
    ]
    table = study_df[cols].copy()

    style = (
        table.style
        .format({
            'MAPE (%)': '{:.3f}', 'MPE (%)': '{:.3f}', 'Max_PE (%)': '{:.3f}',
            'RMSE_raw': '{:.3f}', 'MSE_raw_life': '{:.3f}', 'R2_raw_life': '{:.4f}',
            'MSE': '{:.6f}', 'RMSE': '{:.6f}', 'MAE': '{:.6f}', 'R2 (log)': '{:.4f}',
        }, na_rep='N/A')
        .highlight_min(subset=['MAPE (%)'], color='lightgreen')
        .highlight_max(subset=['Max_PE (%)'], color='lightsalmon')
    )
    display(style)


def plot_study_mape(study_df: pd.DataFrame, title: str):
    if study_df.empty:
        display(Markdown(f"**{title}: no data available for plotting.**"))
        return

    d = study_df[(study_df['target'] == 'LogLife') & study_df['MAPE (%)'].notna()].copy()
    if d.empty:
        display(Markdown(f"**{title}: no LogLife MAPE rows available.**"))
        return

    d['x_key'] = d['regime'].astype(str) + ' | ' + d['ablation'].astype(str)
    x_order = list(dict.fromkeys(d['x_key'].tolist()))
    model_order = [m for m in MODEL_COLORS if m in set(d['model_family'].astype(str))]

    x = np.arange(len(x_order), dtype=float)
    width = 0.8 / max(1, len(model_order))

    fig, ax = plt.subplots(figsize=(max(8, 1.9 * len(x_order)), 4.8))
    for i, model in enumerate(model_order):
        dm = d[d['model_family'].astype(str) == model]
        vals = [
            float(dm.loc[dm['x_key'] == key, 'MAPE (%)'].iloc[0]) if (dm['x_key'] == key).any() else np.nan
            for key in x_order
        ]
        ax.bar(x + (i - (len(model_order) - 1) / 2) * width, vals, width=width, label=model, color=MODEL_COLORS[model])

    ax.set_xticks(x)
    ax.set_xticklabels(x_order, rotation=30, ha='right')
    ax.set_xlabel('Regime | Ablation')
    ax.set_ylabel('MAPE (%)')
    ax.set_title(title)
    ax.legend(title='Model family')
    fig.tight_layout()
    plt.show()


## Study 1: Input Feature Ablation — Uniform Regime

Ablations: `Edge`, `Edge_arc`, `Edge_arc_feat`  
Models: `ArGEnT_self_att_noSDF`, `PointNetMLPJoint`, `PointNetMLPJoint_headfeat`


In [ ]:
study1_df = get_study_results(
    metrics_long,
    regimes=['Uniform'],
    ablations=['Edge', 'Edge_arc', 'Edge_arc_feat'],
    models=['ArGEnT_self_att_noSDF', 'PointNetMLPJoint', 'PointNetMLPJoint_headfeat'],
)


In [ ]:
render_study_table(study1_df, 'Study 1 Summary')


In [ ]:
plot_study_mape(study1_df, 'Study 1 — Uniform input-feature ablation (MAPE)')


- `MAPE (%)` and `Max_PE (%)` are the primary ranking metrics for `LogLife` in this study.
- The table isolates how adding arc-length and derivative features changes life-prediction error under Uniform sampling.
- Use `MPE (%)` to identify systematic over- or under-prediction bias.


## Study 2: Input Feature Ablation — Zonal Regime

Ablations: `Edge`, `Edge_arc`, `Edge_arc_feat`, `Edge_zoneID`  
Models: `ArGEnT_self_att_noSDF`, `PointNetMLPJoint`, `PointNetMLPJoint_headfeat`


In [ ]:
study2_df = get_study_results(
    metrics_long,
    regimes=['Zonal'],
    ablations=['Edge', 'Edge_arc', 'Edge_arc_feat', 'Edge_zoneID'],
    models=['ArGEnT_self_att_noSDF', 'PointNetMLPJoint', 'PointNetMLPJoint_headfeat'],
)


In [ ]:
render_study_table(study2_df, 'Study 2 Summary')


In [ ]:
plot_study_mape(study2_df, 'Study 2 — Zonal input-feature ablation (MAPE)')


- This study adds `Edge_zoneID` to test whether explicit zone encoding helps under Zonal sampling.
- `MAPE (%)` and `Max_PE (%)` indicate whether improvements are robust or driven by outliers.
- Compare `MPE (%)` signs to track bias shifts introduced by each ablation.


## Study 3: Uniform vs. Zonal — Effect of Regime

Fixed ablation: `Edge`  
Models: `ArGEnT_self_att_noSDF`, `PointNetMLPJoint`


In [ ]:
study3_df = get_study_results(
    metrics_long,
    regimes=['Uniform', 'Zonal'],
    ablations=['Edge'],
    models=['ArGEnT_self_att_noSDF', 'PointNetMLPJoint'],
)


In [ ]:
render_study_table(study3_df, 'Study 3 Summary')


In [ ]:
plot_study_mape(study3_df, 'Study 3 — Uniform vs Zonal at Edge baseline (MAPE)')


- This paired comparison quantifies regime difficulty using the same baseline feature set.
- Higher `MAPE (%)`/`Max_PE (%)` in Zonal rows indicates a harder prediction setting.
- Cross-model consistency in the direction of change strengthens the regime-level conclusion.


## Study 4: Effect of Proximity Feature — Edge vs. Edge_Prox

Ablations: `Edge`, `Edge_Prox`  
Regimes: Uniform and Zonal  
Models: `ArGEnT_self_att_noSDF`, `PointNetMLPJoint`


In [ ]:
study4_df = get_study_results(
    metrics_long,
    regimes=['Uniform', 'Zonal'],
    ablations=['Edge', 'Edge_Prox'],
    models=['ArGEnT_self_att_noSDF', 'PointNetMLPJoint'],
)


In [ ]:
render_study_table(study4_df, 'Study 4 Summary')


In [ ]:
plot_study_mape(study4_df, 'Study 4 — Edge vs Edge_Prox across regimes (MAPE)')


- This study isolates the impact of proximity representation relative to the Edge baseline.
- Read Uniform and Zonal rows side-by-side to see if proximity helps consistently by regime.
- `Max_PE (%)` highlights whether proximity mainly reduces extreme local failures.


## Study 5: Stress as Input — Edge vs. Edge_no_stress

Ablations: `Edge`, `Edge_no_stress`  
Regimes: Uniform and Zonal  
Models: `ArGEnT_self_att_noSDF`, `PointNetMLPJoint`


In [ ]:
study5_df = get_study_results(
    metrics_long,
    regimes=['Uniform', 'Zonal'],
    ablations=['Edge', 'Edge_no_stress'],
    models=['ArGEnT_self_att_noSDF', 'PointNetMLPJoint'],
)


In [ ]:
render_study_table(study5_df, 'Study 5 Summary')


In [ ]:
plot_study_mape(study5_df, 'Study 5 — Edge vs Edge_no_stress across regimes (MAPE)')


- This ablation tests how much stress-target coupling contributes to life prediction quality.
- Compare `Edge_no_stress` against `Edge` using `MAPE (%)` as the headline metric.
- Use `MPE (%)` and `Max_PE (%)` to identify bias and worst-case penalties from removing stress.


## Per-Node Percentage Error Visualisation


In [ ]:
def resolve_example_file(filename: str) -> Optional[Path]:
    for root in EXAMPLE_DIR_CANDIDATES:
        p = root / filename
        if p.exists():
            return p
    return None


def load_first_pointset(path: Path) -> np.ndarray:
    with h5py.File(path, 'r') as f:
        if 'samples' not in f or len(f['samples'].keys()) == 0:
            raise RuntimeError(f'No samples in {path}')
        first_key = sorted(f['samples'].keys())[0]
        grp = f['samples'][first_key]
        coords = grp['node_coords_mm'][:].astype(np.float32)
        zone_id = grp['zone_id'][:].astype(np.float32)
        stress = grp['stress_max_vm'][:].astype(np.float32)
        life_raw = grp['life_raw'][:].astype(np.float32)

        cols: List[np.ndarray] = [coords[:, 0], coords[:, 1], zone_id]
        if 'arc_length_mm' in grp:
            arc = grp['arc_length_mm'][:].astype(np.float32)
            if not np.any(np.isnan(arc)):
                cols.append(arc)
        if 'node_features' in grp:
            feats = grp['node_features'][:]
            if feats.ndim == 2 and feats.shape[1] > 0:
                for i in range(feats.shape[1]):
                    cols.append(feats[:, i].astype(np.float32))

        cols.append(stress)
        cols.append(np.log10(np.clip(life_raw, 1e-12, None)).astype(np.float32))
        return np.stack(cols, axis=-1).astype(np.float32)


def model_row_for_example(regime: str, ablation: str, model_family: str) -> Optional[pd.Series]:
    m = selected_models_df[
        (selected_models_df['regime'] == regime)
        & (selected_models_df['ablation'] == ablation)
        & (selected_models_df['model_family'] == model_family)
    ]
    if m.empty:
        return None
    row = m.iloc[0]
    ckpt = row['checkpoint_path']
    if ckpt is None or not Path(ckpt).exists():
        return None
    return row


EXAMPLE_SPECS = [
    ('disc_example_edge_deriv_uniform.h5', 'Uniform', 'Edge'),
    ('disc_example_edge_deriv_zonal.h5', 'Zonal', 'Edge'),
    ('disc_example_edge_proximity_uniform.h5', 'Uniform', 'Edge_Prox'),
    ('disc_example_edge_proximity_zonal.h5', 'Zonal', 'Edge_Prox'),
]

if torch is None or h5py is None:
    warnings.warn('Skipping per-node visualisation because torch and/or h5py are unavailable in this runtime.')
else:
    for filename, regime, ablation in EXAMPLE_SPECS:
        example_path = resolve_example_file(filename)
        if example_path is None:
            warnings.warn(f'Missing example file: {filename} (skipped)')
            continue

        try:
            arr = load_first_pointset(example_path)
        except Exception as exc:
            warnings.warn(f'Failed to load {filename}: {type(exc).__name__}: {exc}')
            continue

        model_rows: List[Tuple[str, pd.Series]] = []
        for family in ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint']:
            mr = model_row_for_example(regime, ablation, family)
            if mr is not None:
                model_rows.append((family, mr))

        if not model_rows:
            warnings.warn(f'No matching checkpoints available for {regime}/{ablation} ({filename})')
            continue

        plot_payload = []
        for family, mr in model_rows:
            try:
                pred, true, target_names, _ = run_model_on_array(arr, mr)
                tnames = [map_target_name(t) for t in target_names]
                if 'LogLife' not in tnames:
                    warnings.warn(f'No LogLife target available for {family} on {filename}')
                    continue
                life_idx = tnames.index('LogLife')
                pred_raw = loglife_to_raw_life(pred[:, life_idx])
                true_raw = loglife_to_raw_life(true[:, life_idx])
                pe = (pred_raw - true_raw) / np.clip(true_raw, 1e-12, None) * 100.0
                stats = percentage_metrics(true_raw, pred_raw)
                plot_payload.append((family, pe, stats))
            except Exception as exc:
                warnings.warn(f'Inference failed for {family} on {filename}: {type(exc).__name__}: {exc}')

        if not plot_payload:
            continue

        vmax = max(float(np.max(np.abs(pe))) for _, pe, _ in plot_payload)
        vmax = max(vmax, 1e-6)
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

        fig, axes = plt.subplots(1, len(plot_payload), figsize=(6.2 * len(plot_payload), 4.8), squeeze=False)
        axes = axes.ravel()

        for ax, (family, pe, _) in zip(axes, plot_payload):
            sc = ax.scatter(arr[:, 0], arr[:, 1], c=pe, cmap='RdBu_r', norm=norm, s=12)
            ax.set_xlabel('x')
            ax.set_ylabel('y')
            ax.set_title(f"{regime} | {ablation} | {family} — Per-Node % Error")
            cbar = fig.colorbar(sc, ax=ax)
            cbar.set_label('Percentage Error (%)')

        fig.tight_layout()
        plt.show()

        stats_rows = [
            {
                'model_family': family,
                'MAPE (%)': stats['MAPE (%)'],
                'MPE (%)': stats['MPE (%)'],
                'Max_PE (%)': stats['Max_PE (%)'],
            }
            for family, _, stats in plot_payload
        ]
        stats_df = pd.DataFrame(stats_rows)
        display(Markdown(f"**{filename} — per-example stats**"))
        display(stats_df.style.format({'MAPE (%)': '{:.3f}', 'MPE (%)': '{:.3f}', 'Max_PE (%)': '{:.3f}'}))
